# Build the FAST Utilities APK in Google Colab

This notebook builds an **installable release APK** for the FAST Utilities Android app using the
Android Gradle Plugin — **no Android Studio and no Expo account / EAS login required**.

**What it does:**
1. Installs **JDK 17**, **Node 20** and the **Android SDK** (API 36, NDK 27.1, CMake 3.30.5).
2. Clones this repo and installs its dependencies.
3. Runs `expo prebuild` to generate the native `android/` project.
4. Builds `app-release.apk` with Gradle.
5. Downloads the APK to your machine (optionally saves a copy to Google Drive).

**Notes**
- The first build takes **~10–20 minutes** (NDK + Gradle downloads). Later runs are faster.
- The APK is signed with the Expo **debug keystore** by default, so it is fine to **sideload /
  install directly** on a device, but it is **not** for Google Play. For a Play Store AAB you must
  configure a release keystore (see the README) or use `eas build -p android --profile production`.
- Run the cells **top to bottom**. If a cell fails, re-run it.


In [ ]:
# ── 1. Install JDK 17 (required by Android Gradle Plugin 8.12 / Kotlin 2.1) ──
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless unzip curl > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

!$JAVA_HOME/bin/java -version


In [ ]:
# ── 2. Install Node.js 20 (Expo SDK 57 requires Node 20+) ──
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null

!node --version
!npm --version


In [ ]:
# ── 3. Install the Android SDK and the exact packages this project pins ──
# Versions come from node_modules/react-native/ReactAndroid/gradle/libs.versions.toml:
#   compileSdk/targetSdk = 36, build-tools = 36.0.0, ndk = 27.1.12297006, cmake = 3.30.5
import os
os.environ["ANDROID_HOME"] = "/opt/android-sdk"
os.environ["ANDROID_SDK_ROOT"] = "/opt/android-sdk"

!mkdir -p $ANDROID_HOME/cmdline-tools
!curl -fsSL -o /tmp/cmdtools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
!unzip -q -o /tmp/cmdtools.zip -d $ANDROID_HOME/cmdline-tools
!mv $ANDROID_HOME/cmdline-tools/cmdline-tools $ANDROID_HOME/cmdline-tools/latest

# Accept all licenses, then install the SDK components (NDK is the big one, ~2–3 min)
!yes | $ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager \
    "platform-tools" \
    "platforms;android-36" \
    "build-tools;36.0.0" \
    "ndk;27.1.12297006" \
    "cmake;3.30.5"


In [ ]:
# ── 4. Clone this repo and install dependencies ──
!rm -rf /content/fast-utilities-android-v1
!git clone --depth 1 https://github.com/ammarasad2005/fast-utilities-android-v1.git /content/fast-utilities-android-v1

%cd /content/fast-utilities-android-v1
!npm ci


In [ ]:
# ── 5. Generate the native Android project (expo prebuild) ──
%cd /content/fast-utilities-android-v1
!npx expo prebuild --platform android --no-install

# Point Gradle at the SDK + JDK we installed and give it enough memory
!echo "sdk.dir=/opt/android-sdk" > android/local.properties
!echo "org.gradle.java.home=/usr/lib/jvm/java-17-openjdk-amd64" >> android/gradle.properties
!echo "org.gradle.jvmargs=-Xmx4096m -XX:MaxMetaspaceSize=1024m" >> android/gradle.properties

!echo "--- generated android/ folder ---"
!ls android


In [ ]:
# ── 6. Build the release APK (10–15 min on first run) ──
%cd /content/fast-utilities-android-v1/android
!./gradlew assembleRelease --no-daemon


In [ ]:
# ── 7. Download the APK ──
import os

apk = "/content/fast-utilities-android-v1/android/app/build/outputs/apk/release/app-release.apk"
print("APK exists:", os.path.exists(apk))
if os.path.exists(apk):
    print("Size: %.1f MB" % (os.path.getsize(apk) / 1e6))

from google.colab import files
files.download(apk)

# Optional: also save a copy to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "$apk" /content/drive/MyDrive/fast-utilities-v1.apk
